In [1]:
MODEL = "gpt-5.6-luna"

# Price per 1 million tokens
INPUT_PRICE = 0.20
OUTPUT_PRICE = 1.20

# LOAD QUESTIONS
import pandas as pd

df = pd.read_csv("ai_teacher_20_questions.csv")
print(f"Total questions: {len(df)}")

import os
from dotenv import load_dotenv
from openai import OpenAI
import time

# 1. LOAD API KEY

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# AI TEACHER PROMPT

instructions = """
You are an expert AI Teacher.

Explain the concept using simple layman terms.

Keep the answer precise and easy to understand.

Give the answer in approximately 3 lines.
"""

# STORAGE

answers = []
latencies = []
input_tokens = []
output_tokens = []
total_tokens = []
input_costs = []
output_costs = []
total_costs = []


# GENERATE ANSWERS

for index, row in df.iterrows():

    question = row["User Query"]

    try:
        # START TIMER
        start_time = time.perf_counter()

        # OPENAI API CALL
        response = client.responses.create(model=MODEL, instructions=instructions, input=question)

        # END TIMER
        end_time = time.perf_counter()

        # LATENCY
        latency = end_time - start_time

        # ANSWER
        answer = response.output_text.strip()

        # TOKEN USAGE
        usage = response.usage
        input_token_count = usage.input_tokens
        output_token_count = usage.output_tokens
        total_token_count = usage.total_tokens

        # COST
        input_cost = (input_token_count / 1_000_000) * INPUT_PRICE
        output_cost = (output_token_count / 1_000_000) * OUTPUT_PRICE
        total_cost = input_cost + output_cost

        # STORE VALUES
        answers.append(answer)
        latencies.append(latency)
        input_tokens.append(input_token_count)
        output_tokens.append(output_token_count)
        total_tokens.append(total_token_count)
        input_costs.append(input_cost)
        output_costs.append(output_cost)
        total_costs.append(total_cost)


    except Exception as e:
        print(e)

        answers.append("ERROR")
        latencies.append(0)
        input_tokens.append(0)
        output_tokens.append(0)
        total_tokens.append(0)
        input_costs.append(0)
        output_costs.append(0)
        total_costs.append(0)

# ADD RESULTS TO DATAFRAME

df["GeneratedAnswer"] = answers
df["Latency"] = latencies
df["InputTokens"] = input_tokens
df["OutputTokens"] = output_tokens
df["TotalTokens"] = total_tokens
df["InputCost"] = input_costs
df["OutputCost"] = output_costs
df["TotalCost"] = total_costs

# SAVE RESULTS

df.to_csv("ai_teacher_evaluation_Operational_metrics.csv",index=False)

# DISPLAY DATAFRAME

df[
    [
        "User Query",
        "GeneratedAnswer",
        "Latency",
        "InputTokens",
        "OutputTokens",
        "TotalTokens",
        "InputCost",
        "OutputCost",
        "TotalCost"
    ]
]

Total questions: 20


,User Query,GeneratedAnswer,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is technology tha...,5.695166,49,54,103,0.000010,0.000065,0.000075
1,How does AI differ from traditional programming?,Traditional programming follows explicit rules...,2.230081,53,58,111,0.000011,0.000070,0.000080
2,What is Machine Learning?,Machine learning is a type of artificial intel...,2.243544,50,65,115,0.000010,0.000078,0.000088
3,What is Deep Learning?,Deep learning is a type of artificial intellig...,2.658339,50,67,117,0.000010,0.000080,0.000090
4,"What is the difference between AI, ML, and Dee...",- **AI (Artificial Intelligence):** The broad ...,1.873968,58,83,141,0.000012,0.000100,0.000111
5,What is training data?,Training data is the information used to teach...,2.150725,50,57,107,0.000010,0.000068,0.000078
6,What is a Machine Learning model?,A machine learning model is a computer program...,1.792473,52,54,106,0.000010,0.000065,0.000075
7,What is supervised learning?,Supervised learning is a type of machine learn...,2.006094,50,67,117,0.000010,0.000080,0.000090
8,What is unsupervised learning?,Unsupervised learning is a type of machine lea...,2.058075,52,56,108,0.000010,0.000067,0.000078
9,What is reinforcement learning?,Reinforcement learning is a type of machine le...,1.668529,50,51,101,0.000010,0.000061,0.000071


In [8]:
# Load the NEW results
df = pd.read_csv("ai_teacher_evaluation_Operational_metrics.csv")

# CONVERT METRIC COLUMNS TO NUMERIC

metric_columns = [
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]

for column in metric_columns:

    df[column] = pd.to_numeric(df[column], errors="coerce")

# AVERAGE METRICS

average_latency = df["Latency"].mean()
average_input_tokens = df["InputTokens"].mean()
average_output_tokens = df["OutputTokens"].mean()
average_total_tokens = df["TotalTokens"].mean()
average_input_cost = df["InputCost"].mean()
average_output_cost = df["OutputCost"].mean()
average_total_cost = df["TotalCost"].mean()

# TOTAL METRICS

total_latency = df["Latency"].sum()
total_input_tokens = df["InputTokens"].sum()
total_output_tokens = df["OutputTokens"].sum()
total_tokens = df["TotalTokens"].sum()
total_input_cost = df["InputCost"].sum()
total_output_cost = df["OutputCost"].sum()
total_cost = df["TotalCost"].sum()

# CREATE OPERATIONAL METRICS DATAFRAME

operational_metrics_df = pd.DataFrame({

    "Metric": [
        "Latency (seconds)",
        "Input Tokens",
        "Output Tokens",
        "Total Tokens",
        "Input Cost (USD)",
        "Output Cost (USD)",
        "Total Cost (USD)"
    ],

    "Average": [
        average_latency,
        average_input_tokens,
        average_output_tokens,
        average_total_tokens,
        average_input_cost,
        average_output_cost,
        average_total_cost
    ],

    "Total": [
        total_latency,
        total_input_tokens,
        total_output_tokens,
        total_tokens,
        total_input_cost,
        total_output_cost,
        total_cost
    ]
})


# ROUND VALUES

operational_metrics_df["Average"] = (operational_metrics_df["Average"].round(6))
operational_metrics_df["Total"] = (operational_metrics_df["Total"].round(6))

# DISPLAY DATAFRAME
operational_metrics_df

,Metric,Average,Total
0,Latency (seconds),2.072458,41.449158
1,Input Tokens,51.050000,1021.000000
2,Output Tokens,63.050000,1261.000000
3,Total Tokens,114.100000,2282.000000
4,Input Cost (USD),0.000010,0.000204
5,Output Cost (USD),0.000076,0.001513
6,Total Cost (USD),0.000086,0.001717


In [9]:
operational_metrics_df.to_csv(
    "ai_teacher_evaluation_Operational_metrics_average.csv",
    index=False,
    encoding="utf-8-sig"
)